# Notebook 01 — Environment Setup & TargetDiff Test

Day 1 goals:
1. Install / verify all dependencies
2. Clone TargetDiff and download pretrained checkpoint
3. Run a quick sanity-generation on a toy pocket
4. Confirm RDKit, Vina, and py3Dmol are working


In [ ]:
# ── Colab: mount Drive and set up project path ──────────────────────────────
import os, sys

# If running on Colab, uncomment the block below:
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_ROOT = '/content/drive/MyDrive/egfr_diffusion'
# os.makedirs(PROJECT_ROOT, exist_ok=True)
# os.chdir(PROJECT_ROOT)
# !git clone https://github.com/<YOUR_GITHUB>/egfr_diffusion.git . || git pull

# Local run:
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print('PROJECT_ROOT:', PROJECT_ROOT)

In [ ]:
# ── Install dependencies (uncomment on Colab) ───────────────────────────────
# !pip install rdkit biopython umap-learn py3Dmol vina tqdm pyyaml seaborn
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118
# !pip install torch-geometric torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.0.0+cu118.html

In [ ]:
# ── Version check ───────────────────────────────────────────────────────────
import rdkit; print('RDKit:', rdkit.__version__)
import torch;  print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
import numpy;  print('NumPy:', numpy.__version__)
import pandas; print('Pandas:', pandas.__version__)

In [ ]:
# ── Clone TargetDiff (first time only) ──────────────────────────────────────
import subprocess, os
targetdiff_dir = os.path.join(PROJECT_ROOT, 'external', 'targetdiff')

if not os.path.exists(targetdiff_dir):
    subprocess.run([
        'git', 'clone',
        'https://github.com/guanjq/targetdiff.git',
        targetdiff_dir
    ], check=True)
    print('TargetDiff cloned.')
else:
    print('TargetDiff already exists.')

if targetdiff_dir not in sys.path:
    sys.path.insert(0, targetdiff_dir)

In [ ]:
# ── Download pretrained checkpoint ──────────────────────────────────────────
# The official checkpoint is released at:
# https://drive.google.com/file/d/1_BUWcHMQLbvOPbU4aYiDYcvF_0VEPjPZ
#
# On Colab, use gdown:
# !pip install gdown
# import gdown
# ckpt_dir = os.path.join(targetdiff_dir, 'checkpoints')
# os.makedirs(ckpt_dir, exist_ok=True)
# gdown.download('https://drive.google.com/uc?id=1_BUWcHMQLbvOPbU4aYiDYcvF_0VEPjPZ',
#                os.path.join(ckpt_dir, 'pretrained_diffusion.pt'), quiet=False)
print('Checkpoint path: external/targetdiff/checkpoints/pretrained_diffusion.pt')
print('Manual download: https://drive.google.com/file/d/1_BUWcHMQLbvOPbU4aYiDYcvF_0VEPjPZ')

In [ ]:
# ── Quick RDKit smoke test ───────────────────────────────────────────────────
from rdkit import Chem
from rdkit.Chem import QED, Draw
from IPython.display import display

erlotinib_smiles = 'C(#N)c1cc2c(cc1OCC)NC(=O)c1cc(OCC)c(OCC)c(OC)c1N2'
mol = Chem.MolFromSmiles(erlotinib_smiles)
assert mol is not None, 'RDKit parse failed'
print(f'Erlotinib QED: {QED.qed(mol):.3f}  (expect ~0.55)')
display(Draw.MolToImage(mol, size=(300, 200)))

In [ ]:
# ── Verify src module imports ────────────────────────────────────────────────
from src import MolEvaluator, load_config
cfg = load_config(os.path.join(PROJECT_ROOT, 'configs', 'targets.yaml'))
print('Targets config loaded:', list(cfg['targets'].keys()))
print('All imports OK ✓')